In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image


# -------------------------------------------------
# 1. Load pretrained low-compute segmentation model
# -------------------------------------------------
def load_segmentation_model():
    tflite_path = "/mnt/Personal/Projects/Autofocus/Code/Object_Extraction/deep/deeplab_ade20k.tflite"
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    input_details = interpreter.get_input_details()
    interpreter.allocate_tensors()
    input_size = input_details[0]['shape'][2], input_details[0]['shape'][1]
    print(input_size)
    
    return model


# -------------------------------------------------
# 2. Resize function with correct interpolation
# -------------------------------------------------
def resize_image_matrix(image, target_size):
    """
    Resizes image to target size.
    Uses:
    - Bilinear interpolation for upscaling (better smoothness).
    - Area interpolation for downscaling (avoids aliasing).
    """
    h, w = image.shape[:2]

    if h < target_size or w < target_size:
        # Upscale
        resized = tf.image.resize(
            image,
            (target_size, target_size),
            method=tf.image.ResizeMethod.BILINEAR
        )
    else:
        # Downscale
        resized = tf.image.resize(
            image,
            (target_size, target_size),
            method=tf.image.ResizeMethod.AREA
        )

    return tf.cast(resized, tf.uint8)


# -------------------------------------------------
# 3. Preprocess image patch
# -------------------------------------------------
def preprocess_patch(image_path, target_size):
    image = Image.open(image_path).convert("RGB")
    image = np.array(image)

    resized = resize_image_matrix(image, target_size)
    input_tensor = tf.expand_dims(resized, axis=0)

    return image, input_tensor


# -------------------------------------------------
# 4. Run segmentation
# -------------------------------------------------
def segment_image(model, input_tensor):
    result = model(input_tensor)
    mask = result["default"][0]
    return mask


# -------------------------------------------------
# 5. Resize mask back to original resolution
# -------------------------------------------------
def resize_mask(mask, original_shape):
    mask_resized = tf.image.resize(
        mask[..., tf.newaxis],
        (original_shape[0], original_shape[1]),
        method="nearest"
    )
    return tf.squeeze(mask_resized)


# -------------------------------------------------
# 6. Plot original and segmented output
# -------------------------------------------------
def plot_results(original, mask):
    plt.figure(figsize=(10, 5))

    plt.subplot(1, 2, 1)
    plt.title("Original")
    plt.imshow(original)
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.title("Segmentation")
    plt.imshow(mask)
    plt.axis("off")

    plt.show()


# -------------------------------------------------
# 7. Full pipeline
# -------------------------------------------------
def run_segmentation(image_path, target_size=257):
    model = load_segmentation_model()
    original, input_tensor = preprocess_patch(image_path, target_size)

    mask = segment_image(model, input_tensor)
    mask_resized = resize_mask(mask, original.shape)

    plot_results(original, mask_resized)


# Example usage
run_segmentation("/mnt/Personal/Projects/Autofocus/test_folder/sofa.jpg")

2026-03-03 00:38:42.415396: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-03 00:38:42.433619: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772478522.453561  494882 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772478522.459799  494882 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772478522.474964  494882 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

ModuleNotFoundError: No module named 'tensorflow_hub'